# Multiple Instance Learning for single-cell data

This tutorial demonstrates how to run **Multiple Instance Learning (MIL)** models for patient-level prediction from single-cell data using `patpy`.

Each patient is treated as a *bag* of cells. MIL models learn a weighted aggregation over those cells to predict a patient-level label. We cover:

1. Data loading, QC and splits
2. Running a MIL benchmark (`MixMIL`, `ABMIL`, `TransMIL`, `DSMIL`)
3. Train / val / test performance metrics
4. Patient-level representation UMAPs
5. Representation quality scores (`patpy.tl.evaluate_representation`)
6. Cell-level attention visualisations

## Import packages

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

import patpy
import patpy.pp
import patpy.tl
import patpy.pl
from patpy.tl.supervised import MixMIL
from patpy.tl.mil_models import ABMIL, TransMIL, DSMIL
from patpy.tl.benchmark import MILBenchmark

sc.settings.verbosity = 1


In [ ]:
patpy.__version__

## Read the data

We use the [COMBAT dataset](https://www.kaggle.com/datasets/shitovvladimir/a-blood-atlas-of-covid-19-combat-preprocessed) — 783k cells from 140 COVID-19 patients and healthy donors, pre-processed with PCA.

In [ ]:
ADATA_PATH = "/lustre/groups/aih/dinesh.haridoss/datasets/combat_processed.h5ad"

# Labels to train a separate MIL model for each entry
# Format: {column_name: task}  task = "classification" | "regression"
LABELS = {
    "binary_condition": "classification",
    "Source":           "classification",
    "Outcome":          "classification",
}


In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
adata

## Set key columns

In [ ]:
sample_key    = "scRNASeq_sample_ID"
cell_type_key = "cell_type"
metadata_cols = ["Source", "Outcome", "Death28", "Institute"] + list(LABELS.keys())
metadata_cols = list(dict.fromkeys(metadata_cols))  # deduplicate


In [ ]:
adata.obs.rename(columns={"Annotation_major_subset": cell_type_key}, inplace=True)

# Keep only COVID-19 and healthy donors
adata = adata[~adata.obs["Source"].isin(["Sepsis", "Flu"])]

# Binary label: 1 = COVID-19, 0 = healthy
adata.obs["binary_condition"] = adata.obs["Source"].str.contains("COVID").astype(int)

# Drop rows where any label is missing
for lk in LABELS:
    if lk in adata.obs.columns:
        adata = adata[adata.obs[lk].notna()].copy()

adata.obs["binary_condition"].value_counts()


## QC and filtering

In [ ]:
metadata = (
    adata.obs[[sample_key] + metadata_cols]
    .drop_duplicates()
    .set_index(sample_key)
)
n_cells     = patpy.pp.calculate_n_cells_per_sample(adata, sample_key)
composition = patpy.pp.calculate_compositional_metrics(adata, sample_key, [cell_type_key], normalize_to=100)
metadata = pd.concat([metadata, n_cells.loc[metadata.index], composition.loc[metadata.index]], axis=1)
metadata.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(len(n_cells)), n_cells["n_cells"].sort_values(ascending=False).values,
       color="#4e79a7", width=1)
ax.axhline(250, color="red", linestyle="--", label="threshold (250)")
ax.set_xlabel("Patient (ranked)")
ax.set_ylabel("# cells")
ax.set_title("Cells per patient")
ax.legend()
plt.tight_layout()


In [ ]:
adata = patpy.pp.filter_small_samples(adata, sample_key=sample_key, sample_size_threshold=250)
print(f"After QC: {adata.n_obs:,} cells | {adata.obs[sample_key].nunique()} patients")


In [ ]:
comp_plot = composition.loc[composition.index.isin(adata.obs[sample_key].unique())]
fig, ax = plt.subplots(figsize=(14, 4))
comp_plot.plot.bar(stacked=True, ax=ax, colormap="tab20", legend=True, width=0.9)
ax.set_xticklabels([])
ax.set_xlabel("Patient")
ax.set_ylabel("% cells")
ax.set_title("Cell-type composition per patient")
ax.legend(title="Cell type", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7)
plt.tight_layout()


## Create train / val / test splits

Splits are created at the patient level, stratified by the first label.
The same splits are reused for all labels.

In [ ]:
first_label = list(LABELS.keys())[0]

patpy.pp.make_sample_splits(
    adata,
    sample_key=sample_key,
    label_key=first_label,
    test_size=0.2,
    val_size=0.15,
    n_splits=3,
    seed=42,
)

split_cols = sorted([c for c in adata.obs.columns if c.startswith("split")])
print("Split columns written:", split_cols)


In [ ]:
# Compute cell UMAP once for attention visualisations
if "X_umap" not in adata.obsm:
    sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=15)
    sc.tl.umap(adata)
print("UMAP ready")


## Helper functions

In [ ]:
import inspect as _inspect

def _get_importance(model, label):
    sig = _inspect.signature(model.get_cell_importance)
    if "normalized" in sig.parameters:
        return model.get_cell_importance(label=label, normalized=False)
    return model.get_cell_importance(label=label)


# Max cells per bag to avoid GPU OOM for transformer-based models.
# COMBAT bags can reach ~9000 cells; 2000 keeps memory safe on a 32 GB GPU.
MAX_CELLS = 2000

def make_models(label_key, task):
    return {
        "MixMIL": MixMIL(
            sample_key=sample_key, label_keys=[label_key],
            tasks=[task], layer="X_pca", n_epochs=200,
        ),
        "ABMIL": ABMIL(
            sample_key=sample_key, label_keys=[label_key],
            tasks=[task], layer="X_pca", att_dim=128, n_epochs=200,
        ),
        "TransMIL": TransMIL(
            sample_key=sample_key, label_keys=[label_key],
            tasks=[task], layer="X_pca",
            att_dim=128, n_layers=2, n_heads=4,  # smaller than default to save memory
            n_epochs=200, max_cells_per_bag=MAX_CELLS,
        ),
        "DSMIL": DSMIL(
            sample_key=sample_key, label_keys=[label_key],
            tasks=[task], layer="X_pca", att_dim=128,
            n_epochs=200, max_cells_per_bag=MAX_CELLS,
        ),
    }


## Per-label MIL training and evaluation

A separate set of models is trained for each label in `LABELS`.

In [ ]:
from sklearn.metrics import roc_curve, auc as sk_auc

all_results = {}
fitted_full = {}  # {label: {model_name: model}} — full-data retrains

cls_metrics = ["auroc", "balanced_accuracy", "f1_weighted", "aupr"]
reg_metrics = ["pearson", "r2", "mae"]

for label_key, task in LABELS.items():
    print(f"\n{"="*60}")
    print(f"  Label: {label_key}  |  Task: {task}")
    print(f"{"="*60}")

    # ── Label distribution ──────────────────────────────────────
    sample_labels = adata.obs.groupby(sample_key)[label_key].first().dropna()
    vc = sample_labels.value_counts()
    colors = sns.color_palette("Set2", len(vc))
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    vc.plot.bar(ax=axes[0], color=colors)
    axes[0].set_title(f"Patient label counts — {label_key}")
    axes[0].set_ylabel("# patients")
    axes[0].tick_params(axis="x", rotation=30)
    for bar in axes[0].patches:
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.3,
                     str(int(bar.get_height())), ha="center", va="bottom")
    axes[1].pie(vc.values, labels=[f"{k} ({v})" for k, v in vc.items()],
                autopct="%1.1f%%", colors=colors)
    axes[1].set_title(f"Label proportions — {label_key}")
    plt.suptitle(f"Label: {label_key}", fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── Split overview ───────────────────────────────────────────
    sample_obs = adata.obs[[sample_key, label_key] + split_cols].groupby(sample_key).first()
    fig, axes = plt.subplots(2, len(split_cols), figsize=(5 * len(split_cols), 8), squeeze=False)
    for i, col in enumerate(split_cols):
        counts = sample_obs[col].value_counts().reindex(["train", "val", "test"]).fillna(0)
        counts.plot.bar(ax=axes[0][i], color=["#4e79a7", "#f28e2b", "#e15759"])
        axes[0][i].set_title(f"Split {i}: sample counts")
        axes[0][i].tick_params(axis="x", rotation=0)
        dist = (
            sample_obs.groupby([col, label_key]).size()
            .unstack(label_key, fill_value=0)
            .reindex(["train", "val", "test"]).fillna(0)
        )
        dist.plot.bar(stacked=True, ax=axes[1][i], colormap="Set2")
        axes[1][i].set_title(f"Split {i}: label distribution")
        axes[1][i].tick_params(axis="x", rotation=0)
        axes[1][i].legend(title=label_key, bbox_to_anchor=(1.01, 1),
                          loc="upper left", fontsize=8)
    fig.suptitle(f"Split overview — {label_key}", fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── Benchmark ───────────────────────────────────────────────
    bench = MILBenchmark(
        models=make_models(label_key, task),
        sample_key=sample_key,
        label_keys=[label_key],
        tasks=[task],
        n_splits=3,
        seed=42,
    )
    results = bench.run(adata)
    all_results[label_key] = results

    print("\n--- Summary ---")
    print(bench.summary().to_string()) if not results.empty else print("  No results — all models failed.")

    # ── Eval metrics: val / test ─────────────────────────────────
    use_metrics = cls_metrics if task == "classification" else reg_metrics
    for split_name in ["val", "test"]:
        subset = results[
            (results["eval_split"] == split_name) &
            (results["metric"].isin(use_metrics))
        ]
        if subset.empty:
            continue
        fig, ax = plt.subplots(figsize=(10, 4))
        sns.barplot(data=subset, x="metric", y="value", hue="model",
                    ax=ax, errorbar="sd", capsize=0.1, palette="Set2")
        ax.set_ylim(0, 1.1)
        ax.axhline(0.5, color="grey", linestyle="--", lw=1, alpha=0.5)
        ax.set_title(f"MIL benchmark — {split_name} set  [{label_key}]", fontsize=12)
        ax.set_ylabel("Score")
        ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()

    # ── Full-dataset retrain (MixMIL + ABMIL for attention / UMAP) ─
    mixmil = MixMIL(sample_key=sample_key, label_keys=[label_key],
                    tasks=[task], layer="X_pca", n_epochs=200)
    mixmil.prepare_anndata(adata)

    abmil = ABMIL(sample_key=sample_key, label_keys=[label_key],
                  tasks=[task], layer="X_pca", att_dim=128, n_epochs=200)
    abmil.prepare_anndata(adata)

    fitted_full[label_key] = {"MixMIL": mixmil, "ABMIL": abmil}

    # ── Training loss ─────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    for model, name, color in [(mixmil, "MixMIL", "#4e79a7"), (abmil, "ABMIL", "#f28e2b")]:
        hist = getattr(model, "training_history", None)
        if hist:
            losses = [step["loss"] for step in hist if "loss" in step]
            ax.plot(losses, label=name, lw=1.5, color=color)
    ax.set_xlabel("Optimiser step")
    ax.set_ylabel("Loss")
    ax.set_title(f"Training loss — [{label_key}]")
    ax.legend()
    plt.tight_layout()
    plt.show()

    # ── Patient representation UMAP (MixMIL) ─────────────────────
    extra_cols = [c for c in ["Source", "Outcome"] if c != label_key and c in metadata.columns]
    mixmil.plot_embedding(
        method="UMAP",
        metadata_cols=[label_key] + extra_cols,
        categorical_palette="tab10",
    )
    plt.suptitle(f"Patient UMAP — MixMIL  [{label_key}]", y=1.01)
    plt.tight_layout()
    plt.show()

    # ── Representation quality scores ─────────────────────────────
    distances = mixmil.calculate_distance_matrix()
    eval_cols = {label_key: task}
    for ec in extra_cols:
        eval_cols[ec] = "classification"
    rows = []
    for col, ctask in eval_cols.items():
        target = metadata.loc[mixmil.samples, col].dropna()
        if target.nunique() < 2:
            continue
        try:
            res = patpy.tl.evaluate_representation(
                distances, target=target, task=ctask,
                n_neighbors=min(5, len(target) - 1))
            rows.append({"covariate": col, "score": res["score"], "metric": res["metric"]})
        except Exception as e:
            print(f"  rep_quality {col}: {e}")
    if rows:
        rep_df = pd.DataFrame(rows)
        fig, ax = plt.subplots(figsize=(5, max(3, 0.6 * len(rep_df))))
        sns.barplot(data=rep_df, y="covariate", x="score", orient="h", ax=ax, palette="Set2")
        ax.set_xlim(0, 1.05)
        ax.set_title(f"KNN representation quality — MixMIL  [{label_key}]", fontsize=11)
        ax.set_xlabel("KNN score")
        for bar in ax.patches:
            ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                    f"{bar.get_width():.3f}", va="center", fontsize=9)
        plt.tight_layout()
        plt.show()

    # ── ROC curve (binary classification only, first test fold) ───
    if task == "classification":
        test_adata = adata[adata.obs[split_cols[0]] == "test"]
        y_true_series = test_adata.obs.groupby(sample_key)[label_key].first().dropna()
        classes = sorted(y_true_series.unique())
        if len(classes) == 2:
            enc = {c: i for i, c in enumerate(classes)}
            y_true = y_true_series.map(enc).values
            fig, ax = plt.subplots(figsize=(5, 5))
            for model, name in [(mixmil, "MixMIL"), (abmil, "ABMIL")]:
                try:
                    preds = model.predict_on_adata(test_adata, label_key)
                    prob_cols = [c for c in preds.columns if c.startswith("prob_")]
                    if len(prob_cols) < 2:
                        continue
                    proba = preds[prob_cols[1]].reindex(y_true_series.index).values
                    fpr, tpr, _ = roc_curve(y_true, proba)
                    ax.plot(fpr, tpr, lw=2, label=f"{name}  AUC={sk_auc(fpr, tpr):.3f}")
                except Exception as e:
                    print(f"  ROC {name}: {e}")
            ax.plot([0, 1], [0, 1], "k--", lw=1)
            ax.set_xlabel("FPR")
            ax.set_ylabel("TPR")
            ax.set_title(f"ROC — {label_key}  (test fold 0)")
            ax.legend(loc="lower right")
            plt.tight_layout()
            plt.show()

    # ── Attention plots (MixMIL + ABMIL) ─────────────────────────
    for model, name in [(mixmil, "MixMIL"), (abmil, "ABMIL")]:
        try:
            fig = patpy.pl.mil.plot_attention_umap(
                adata, model, label_key,
                cell_type_key=cell_type_key, umap_key="X_umap",
                sample_key=sample_key, normalized=False,
                title_prefix=f"{name}: ",
            )
            plt.suptitle(f"Attention UMAP — {name}  [{label_key}]", y=1.01)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"  attention_umap {name}: {e}")

        try:
            fig = patpy.pl.mil.plot_attention_by_cell_type(
                adata, model, label_key,
                cell_type_key=cell_type_key, sample_key=sample_key, normalized=False,
            )
            plt.suptitle(f"Attention by cell type — {name}  [{label_key}]", y=1.01)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"  attention_celltype {name}: {e}")

        try:
            fig = patpy.pl.mil.plot_attention_celltype_heatmap(
                adata, model, label_key,
                cell_type_key=cell_type_key, sample_key=sample_key, normalized=False,
            )
            plt.suptitle(f"Attention heatmap — {name}  [{label_key}]", y=1.01)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"  attention_heatmap {name}: {e}")

        try:
            att_df = _get_importance(model, label_key)
            src_col = f"{label_key}_importance"
            dst_col = f"{name}_{label_key}_attention"
            if src_col in att_df.columns:
                adata.obs[dst_col] = att_df[src_col].values
                print(f"  → adata.obs['{dst_col}'] written")
        except Exception as e:
            print(f"  save_attention {name}: {e}")

print("\nAll labels complete.")


## Representation-based benchmark (unsupervised + linear head)

Unsupervised methods (Pseudobulk, CellGroupComposition, GroupedPseudobulk) and supervised feature-extractors (MixMIL) are evaluated by fitting a logistic-regression / ridge probe on train representations and predicting on val / test.

In [ ]:
from patpy.tl import RepresentationBenchmark
from patpy.tl import Pseudobulk, CellGroupComposition, GroupedPseudobulk

rep_cls_metrics = ["auroc", "balanced_accuracy", "f1_weighted", "aupr"]
rep_reg_metrics = ["pearson", "r2", "mae"]

all_rep_results = {}

for label_key, task in LABELS.items():
    print(f"\n{'='*60}")
    print(f"  Rep-Benchmark  Label: {label_key}  |  Task: {task}")
    print(f"{'='*60}")

    rep_models = {
        "Pseudobulk": Pseudobulk(
            sample_key=sample_key, cell_group_key=cell_type_key, layer="X_pca"),
        "CellComposition": CellGroupComposition(
            sample_key=sample_key, cell_group_key=cell_type_key),
        "GroupedPseudobulk": GroupedPseudobulk(
            sample_key=sample_key, cell_group_key=cell_type_key, layer="X_pca"),
        "MixMIL (reps)": MixMIL(
            sample_key=sample_key, label_keys=[label_key],
            tasks=[task], layer="X_pca", n_epochs=200),
        "ABMIL (reps)": ABMIL(
            sample_key=sample_key, label_keys=[label_key],
            tasks=[task], layer="X_pca", att_dim=128, n_epochs=200),
    }

    rep_bench = RepresentationBenchmark(
        models=rep_models,
        sample_key=sample_key,
        label_keys=[label_key],
        tasks=[task],
        n_splits=3,
        seed=42,
    )
    rep_results = rep_bench.run(adata)
    all_rep_results[label_key] = rep_results

    if rep_results.empty:
        print("  No results — all rep models failed.")
        continue

    print("\n--- Summary ---")
    print(rep_bench.summary().to_string())

    # ── Metrics barplot ──────────────────────────────────────────
    use_metrics = rep_cls_metrics if task == "classification" else rep_reg_metrics
    for split_name in ["val", "test"]:
        subset = rep_results[
            (rep_results["eval_split"] == split_name) &
            (rep_results["metric"].isin(use_metrics))
        ]
        if subset.empty:
            continue
        fig, ax = plt.subplots(figsize=(11, 4))
        sns.barplot(data=subset, x="metric", y="value", hue="model",
                    ax=ax, errorbar="sd", capsize=0.1, palette="Set2")
        ax.set_ylim(0, 1.1)
        ax.axhline(0.5, color="grey", linestyle="--", lw=1, alpha=0.5)
        ax.set_title(f"Rep-benchmark — {split_name} set  [{label_key}]", fontsize=12)
        ax.set_ylabel("Score")
        ax.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()

print("\nAll labels complete.")


## Combined results across labels

In [ ]:
all_df = pd.concat(
    [df.assign(label_trained=lk) for lk, df in all_results.items()],
    ignore_index=True
)

test_all = all_df[
    (all_df["eval_split"] == "test") &
    (all_df["metric"] == "auroc")
]

if not test_all.empty:
    fig, ax = plt.subplots(figsize=(max(8, 2 * len(LABELS)), 5))
    sns.barplot(data=test_all, x="label_trained", y="value", hue="model",
                ax=ax, errorbar="sd", capsize=0.1, palette="Set2")
    ax.set_ylim(0, 1.1)
    ax.set_title("Test AUROC across all labels", fontsize=13)
    ax.set_ylabel("AUROC")
    ax.set_xlabel("Label")
    ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

all_df.groupby(["label_trained", "model", "eval_split", "metric"])["value"].agg(["mean", "std"]).round(4)


## Paper-ready summary table

Mean ± std on the **test set** across all 3 cross-validation splits, for each model and label.
Rows are sorted by label then metric for easy copy-paste into a paper or supplementary table.


In [ ]:
def _summary_table(results_dict, split="test", metrics=None):
    """Pivot benchmark results into a paper-ready mean ± std table."""
    frames = []
    for label, df in results_dict.items():
        sub = df[df["eval_split"] == split].copy()
        if metrics:
            sub = sub[sub["metric"].isin(metrics)]
        sub = sub.assign(label=label)
        frames.append(sub)
    if not frames:
        return pd.DataFrame()
    combined = pd.concat(frames, ignore_index=True)
    agg = (
        combined.groupby(["label", "model", "metric"])["value"]
        .agg(["mean", "std"])
        .round(3)
    )
    agg["mean ± std"] = agg["mean"].map("{:.3f}".format) + " ± " + agg["std"].map("{:.3f}".format)
    table = agg["mean ± std"].unstack("metric")
    return table

# MIL benchmark — test set
print("=== MIL Benchmark (test set) ===")
mil_table = _summary_table(all_results, split="test",
                           metrics=["auroc", "aupr", "balanced_accuracy", "f1_weighted"])
display(mil_table)

# Representation benchmark — test set
print("\n=== Representation Benchmark (test set) ===")
rep_table = _summary_table(all_rep_results, split="test",
                           metrics=["auroc", "aupr", "balanced_accuracy", "f1_weighted"])
display(rep_table)


## Patient representation UMAPs — all methods

For each label we compare patient-level UMAPs from all representation methods side by side.
`model.plot_embedding()` computes a UMAP of the pairwise distance matrix on the bag embeddings.
Unsupervised methods (Pseudobulk, CellComposition, GroupedPseudobulk) are fit on the full dataset.
MixMIL and ABMIL use the full-data retrains from `fitted_full`.


In [ ]:
from patpy.tl import Pseudobulk, CellGroupComposition, GroupedPseudobulk

# Build unsupervised methods once (transductive — run on full adata)
unsup_methods = {
    "Pseudobulk": Pseudobulk(
        sample_key=sample_key, cell_group_key=cell_type_key, layer="X_pca"),
    "CellComposition": CellGroupComposition(
        sample_key=sample_key, cell_group_key=cell_type_key),
    "GroupedPseudobulk": GroupedPseudobulk(
        sample_key=sample_key, cell_group_key=cell_type_key, layer="X_pca"),
}
for name, m in unsup_methods.items():
    m.calculate_distance_matrix(adata)

for label_key in LABELS:
    sup_methods = {
        "MixMIL": fitted_full[label_key]["MixMIL"],
        "ABMIL":  fitted_full[label_key]["ABMIL"],
    }
    all_methods = {**unsup_methods, **sup_methods}
    n_methods = len(all_methods)

    fig, axes = plt.subplots(1, n_methods, figsize=(4 * n_methods, 4), squeeze=False)
    axes = axes[0]

    for ax, (name, model) in zip(axes, all_methods.items()):
        try:
            model.plot_embedding(
                method="UMAP",
                metadata_cols=[label_key],
                categorical_palette="tab10",
                axes=ax,
            )
            ax.set_title(name, fontsize=11)
            ax.set_xlabel("")
            ax.set_ylabel("")
        except Exception as e:
            ax.set_title(f"{name}\n({e})", fontsize=9)

    fig.suptitle(f"Patient representation UMAPs — {label_key}", fontsize=13)
    plt.tight_layout()
    plt.show()


## Summary

| Step | patpy function |
|------|---------------|
| QC — filter small patients | `patpy.pp.filter_small_samples` |
| QC — cell counts & composition | `patpy.pp.calculate_n_cells_per_sample`, `calculate_compositional_metrics` |
| Stratified splits | `patpy.pp.make_sample_splits` |
| Cross-validated benchmark (per label) | `patpy.tl.MILBenchmark` |
| Patient representation UMAP | `model.plot_embedding` |
| Representation quality | `patpy.tl.evaluate_representation` |
| Attention on cell UMAP | `patpy.pl.mil.plot_attention_umap` |
| Attention by cell type | `patpy.pl.mil.plot_attention_by_cell_type` |
| Attention heatmap | `patpy.pl.mil.plot_attention_celltype_heatmap` |
